# 0027 / 02 Winner-only slice

Keep complete strict-win trajectories and emit only the winning player perspective. The default exact-deck hash preserves the 0025 canonical contract.


In [ ]:
from __future__ import annotations
import csv, gzip, hashlib, json, re, zipfile
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Any, Mapping

DATA_START = date(2026, 7, 15)
DATA_END = date(2026, 8, 1)
SELECTED_DATES: set[date] | None = None  # Set e.g. {date(2026, 7, 20)} after inventory review.
TARGET_DECK_SHA256 = "f50fa3a23cdf21be7cf7d3f558b8ff0b82e8d4e7ba8f61b7b4cacc1a0080c16a"
SCHEMA_VERSION = "0025_canonical_semantic_decision_v2"
DATE_RE = re.compile(r"20\d\d-\d\d-\d\d")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_csv(path: Path, rows: list[Mapping[str, Any]], fields: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def path_date(path: Path) -> date | None:
    for value in DATE_RE.findall(str(path)):
        try:
            parsed = date.fromisoformat(value)
        except ValueError:
            continue
        if DATA_START <= parsed <= DATA_END:
            return parsed
    return None

def input_files(root: Path = Path("/kaggle/input")) -> list[Path]:
    return sorted(path for path in root.rglob("*") if path.is_file() and path_date(path) is not None)

def payloads(path: Path):
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as bundle:
            for member in bundle.namelist():
                if not member.lower().endswith(".json"):
                    continue
                try:
                    value = json.loads(bundle.read(member))
                except Exception:
                    continue
                if isinstance(value, dict):
                    yield f"{path}!{member}", value
        return
    try:
        opener = gzip.open if path.name.lower().endswith(".gz") else open
        with opener(path, "rt", encoding="utf-8-sig") as handle:
            value = json.load(handle)
        if isinstance(value, dict):
            yield str(path), value
        elif isinstance(value, list):
            for index, item in enumerate(value):
                if isinstance(item, dict):
                    yield f"{path}#{index}", item
    except Exception:
        return

def first(value: Mapping[str, Any], *keys: str, default: Any = None) -> Any:
    for key in keys:
        if key in value and value[key] is not None:
            return value[key]
    return default

def as_list(value: Any) -> list[Any]:
    return value if isinstance(value, list) else []

def as_int(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        return default

def episode_id(payload: Mapping[str, Any], fallback: str) -> str:
    info = payload.get("info")
    if isinstance(info, Mapping):
        value = first(info, "EpisodeId", "episode_id", "episodeId")
        if value is not None:
            return str(value)
    return str(first(payload, "episode_id", "episodeId", "id", default=fallback))

def winner(payload: Mapping[str, Any]) -> int | None:
    rewards = first(payload, "rewards", "reward", "scores", default=[])
    if not isinstance(rewards, list):
        return None
    indexes = [i for i, value in enumerate(rewards) if isinstance(value, (int, float)) and not isinstance(value, bool) and value > 0]
    return indexes[0] if len(indexes) == 1 else None

def deck_hash(cards: Any) -> str | None:
    ids = []
    for value in as_list(cards):
        if isinstance(value, Mapping):
            value = first(value, "id", "cardId")
        try:
            ids.append(int(value))
        except (TypeError, ValueError):
            return None
    if len(ids) != 60:
        return None
    return hashlib.sha256(",".join(map(str, sorted(ids))).encode("ascii")).hexdigest()

def deck_counts(cards: Any) -> list[list[int]]:
    ids = []
    for value in as_list(cards):
        if isinstance(value, Mapping):
            value = first(value, "id", "cardId")
        try:
            ids.append(int(value))
        except (TypeError, ValueError):
            return []
    counts = Counter(ids)
    return [[identity, counts[identity]] for identity in sorted(counts)]

def frames(payload: Mapping[str, Any]):
    values = first(payload, "steps", "frames", "trajectory", "observations", default=[])
    for index, frame in enumerate(as_list(values)):
        if not isinstance(frame, Mapping):
            continue
        observation = first(frame, "observation", "obs", "state", default=frame)
        if not isinstance(observation, Mapping) or not isinstance(observation.get("select"), Mapping):
            continue
        current = observation.get("current") if isinstance(observation.get("current"), Mapping) else {}
        actor = as_int(first(frame, "playerIndex", "player", "actor", default=current.get("yourIndex", 0)))
        action = first(frame, "action", "selected", "ordered_action", "selection", default=[])
        yield index, actor, observation, action

INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working/ptcg_0027_winner_slice")
OUT.mkdir(parents=True, exist_ok=True)
inventory = next(INPUT.rglob("source_inventory.csv"), None)
source_rows = list(csv.DictReader(inventory.open(encoding="utf-8"))) if inventory else []
sources = [Path(row["path"]) for row in source_rows if SELECTED_DATES is None or date.fromisoformat(row["date"]) in SELECTED_DATES] if inventory else input_files()
if not sources:
    raise FileNotFoundError("no replay sources selected; inspect notebook 01 inventory and adjust SELECTED_DATES")
episodes, decisions = 0, 0
episode_index = []
with gzip.open(OUT / "winner_raw.jsonl.gz", "wt", encoding="utf-8") as output:
    for source in sources:
        for payload_name, payload in payloads(source):
            actor = winner(payload)
            if actor is None:
                continue
            eid = episode_id(payload, payload_name)
            players = as_list(payload.get("players"))
            deck = first(players[actor], "deck", "deck_cards", "registeredDeck", default=[]) if actor < len(players) and isinstance(players[actor], Mapping) else []
            dhash = deck_hash(deck)
            if dhash is None or (TARGET_DECK_SHA256 and dhash != TARGET_DECK_SHA256):
                continue
            count = 0
            for frame_index, player, observation, action in frames(payload):
                if player != actor:
                    continue
                select = observation["select"]
                options = as_list(select.get("option"))
                ordered = [as_int(value, -1) for value in as_list(action)]
                minimum = as_int(select.get("minCount"))
                maximum = max(minimum, as_int(select.get("maxCount"), len(options)))
                if not options or not minimum <= len(ordered) <= maximum or len(set(ordered)) != len(ordered) or any(index < 0 or index >= len(options) for index in ordered):
                    continue
                row = {"schema_version": "0019_universal_winner_decision_v1", "identity": {"date": str(path_date(source)), "episode_id": eid, "player_index": actor, "episode_step": frame_index}, "split": "validation" if int(hashlib.sha256(eid.encode()).hexdigest()[:8], 16) % 10 == 0 else "train", "deck_manifest": {"cards": deck, "counts": deck_counts(deck), "sha256": dhash}, "actor_observation": observation, "legal_options": options, "ordered_action": ordered, "action_termination": True, "event_cursor": {"visual_frame_index": frame_index, "actor_decision_index": count, "incoming_log_count": len(as_list(observation.get("logs")))}, "terminal_outcome": "win", "source_payload_sha256": sha256_file(source) if source.is_file() else "", "source_id": None, "source_team_name": ""}
                output.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")
                count += 1
                decisions += 1
            if count:
                episodes += 1
                episode_index.append({"episode_id": eid, "winner_player": actor, "date": str(path_date(source)), "decisions": count, "deck_sha256": dhash or "", "source": payload_name})
write_csv(OUT / "winner_episode_index.csv", episode_index, ["episode_id", "winner_player", "date", "decisions", "deck_sha256", "source"])
write_json(OUT / "winner_slice_manifest.json", {"schema_version": "0027_winner_slice_v1", "winner_only": True, "strict_reward_positive": True, "data_start": DATA_START.isoformat(), "data_end": DATA_END.isoformat(), "selected_dates": None if SELECTED_DATES is None else sorted(value.isoformat() for value in SELECTED_DATES), "target_deck_sha256": TARGET_DECK_SHA256, "episodes": episodes, "decisions": decisions})
print(json.dumps({"episodes": episodes, "decisions": decisions, "output": str(OUT)}, indent=2))

